# 🏢 Team Modular Multi-Agent System

A **two-level hierarchy**: a *Coordinator* assigns sub-tasks to teams; each team's *Supervisor* routes internally to the best individual agent.

```
 User Task
     │
     ▼
  [Coordinator]  ← with_structured_output → CoordPlan([{team, instruction}, ...])
     │
     ▼  ◄──────────────────────────────────────────────────────┐
  [Team Dispatcher]                                            │ loop
     ├─ research_team → [Team Supervisor] → web_researcher |  │
     │                                      scraper           │
     ├─ data_team     → [Team Supervisor] → analyst |         │
     │                                      visualizer |      │
     │                                      programmer        │
     └─ writing_team  → [Team Supervisor] → writer            │
                                   more steps? ───────────────┘
                                   no → END
```

**Strengths:** Teams can grow independently; clean separation of concerns; easy to extend.  
**Limitations:** Two LLM routing hops add latency; most complex setup of the four patterns.

---
```bash
uv sync --group notebooks
```

## ⚙️ Setup

In [1]:
from typing import TypedDict
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from pydantic import BaseModel

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0)
print("✅ Setup complete.")

✅ Setup complete.


## 🛠️ Tools — Grouped by Domain

Tools are grouped by team domain. Agents only receive their relevant tools, keeping the LLM's choice space small and intent clear.

In [2]:
# ── Research tools ────────────────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"[Web: {query}]"

@tool
def scrape_url(url: str) -> str:
    """Scrape content from a URL."""
    return f"[Page: {url}]"

# ── Data tools ─────────────────────────────────────────────────────────────
@tool
def query_database(sql: str) -> str:
    """Run a SQL query against the database."""
    return f"[DB: {sql}]"

@tool
def generate_chart(spec: str) -> str:
    """Generate a chart from a specification string."""
    return f"[Chart: {spec}]"

@tool
def run_python(code: str) -> str:
    """Execute Python code and return the output."""
    return f"[Exec: {code}]"

# ── Writing tools ──────────────────────────────────────────────────────────
@tool
def write_section(title: str, body: str) -> str:
    """Write a titled report section."""
    return f"[Section '{title}' written]"

print("✅ Tools defined.")

✅ Tools defined.


## 🏗️ Team Registry

Teams are plain dicts. The **Coordinator only knows team names** — it never addresses individual agents directly. This two-level abstraction means you can add or swap agents inside a team without touching the coordinator.

In [3]:
def make_agent(prompt: str, tools: list):
    """Compile a create_agent runnable with a given system prompt and tools."""
    return create_agent(model=llm, tools=tools, system_prompt=prompt)


TEAMS = {
    "research_team": {
        "description": "Handles information gathering and web research.",
        "agents": {
            "web_researcher": make_agent(
                "You are a Web Researcher. Search the web thoroughly.", [search_web]
            ),
            "scraper": make_agent(
                "You are a Web Scraper. Extract content from specific URLs.", [scrape_url]
            ),
        },
    },
    "data_team": {
        "description": "Handles data analysis and visualisation.",
        "agents": {
            "analyst": make_agent(
                "You are a Data Analyst. Query databases and interpret results.", [query_database]
            ),
            "visualizer": make_agent(
                "You are a Visualiser. Create charts and visual summaries.", [generate_chart]
            ),
            "programmer": make_agent(
                "You are a Programmer. Write Python scripts for data processing.", [run_python]
            ),
        },
    },
    "writing_team": {
        "description": "Handles drafting, editing, and document production.",
        "agents": {
            "writer": make_agent(
                "You are a Writer. Produce clear, structured report sections.", [write_section]
            ),
        },
    },
}

print("✅ Teams compiled:")
for tname, tdata in TEAMS.items():
    print(f"  {tname}: {list(tdata['agents'].keys())}")

✅ Teams compiled:
  research_team: ['web_researcher', 'scraper']
  data_team: ['analyst', 'visualizer', 'programmer']
  writing_team: ['writer']


## 🔀 Team Supervisor (Level 2 Routing)

`run_team` uses `llm.with_structured_output(TeamDecision)` to pick the right agent within a team. It also allows the supervisor to **refine** the instruction for that specific agent.

In [4]:
class TeamDecision(BaseModel):
    """The team supervisor's routing decision."""
    agent:       str   # name of the chosen agent within the team
    instruction: str   # refined instruction tailored to that agent's capabilities


def run_team(team_name: str, instruction: str) -> str:
    """Route an instruction through a team supervisor to the best agent."""
    team        = TEAMS[team_name]
    agent_names = list(team["agents"].keys())

    supervisor_llm = llm.with_structured_output(TeamDecision)
    system_msg = (
        f"You are the {team_name} supervisor.\n"
        f"Available agents: {', '.join(agent_names)}\n"
        "Choose the best agent for the instruction and optionally refine the instruction."
    )
    decision_raw = supervisor_llm.invoke([
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": instruction},
    ])
    decision = TeamDecision.model_validate(decision_raw)

    # Safe fallback if the supervisor returns an unknown agent name
    agent = team["agents"].get(decision.agent, list(team["agents"].values())[0])
    result = agent.invoke({"messages": [{"role": "user", "content": decision.instruction}]})
    return result["messages"][-1].content

## 🗂️ Coordinator (Level 1 Routing) & Graph State

The coordinator sees **only teams**, not agents. `llm.with_structured_output(CoordPlan)` returns a validated plan directly.

In [5]:
class CoordStep(BaseModel):
    team:        str   # one of: research_team, data_team, writing_team
    instruction: str   # high-level instruction for that team

class CoordPlan(BaseModel):
    plan: list[CoordStep]


coordinator_llm = llm.with_structured_output(CoordPlan)

COORDINATOR_SYSTEM = (
    "You are a project coordinator. Break the task into ordered steps.\n"
    "Assign each step to one of: research_team, data_team, writing_team.\n"
    "Be concise — avoid redundant steps."
)


class CoordState(TypedDict):
    task:    str         # original user task
    plan:    list        # list[CoordStep]
    current: int         # index of current step
    log:     str         # accumulated output from completed steps

## 🔁 Graph Nodes

In [6]:
def coordinator_node(state: CoordState) -> CoordState:
    """Ask the coordinator to produce a team-level plan."""
    coord_plan_raw = coordinator_llm.invoke([
        {"role": "system", "content": COORDINATOR_SYSTEM},
        {"role": "user",   "content": state["task"]},
    ])
    coord_plan = CoordPlan.model_validate(coord_plan_raw)
    return {**state, "plan": coord_plan.plan, "current": 0}


def dispatch_node(state: CoordState) -> CoordState:
    """Send the current step to its team and collect the output."""
    step   = state["plan"][state["current"]]
    output = run_team(step.team, step.instruction)
    log    = state["log"] + f"\n[{step.team}] → {output}"
    return {**state, "log": log, "current": state["current"] + 1}


def more_steps(state: CoordState) -> str:
    """Loop until all planned steps are dispatched."""
    return "dispatch" if state["current"] < len(state["plan"]) else END

## 🏗️ Build & Compile the Graph

In [7]:
graph = StateGraph(CoordState)
graph.add_node("coordinator", coordinator_node)
graph.add_node("dispatch",    dispatch_node)

graph.set_entry_point("coordinator")
graph.add_edge("coordinator", "dispatch")
graph.add_conditional_edges("dispatch", more_steps)

app = graph.compile()
print("✅ Team Modular graph compiled.")

✅ Team Modular graph compiled.


## ▶️ Demo

A realistic business task that spans all three teams — research, data, and writing.

In [8]:
result = app.invoke({
    "task":    "Research EV market trends, analyse sales data, and write a report with charts.",
    "plan":    [],
    "current": 0,
    "log":     "",
})

print("\n=== TEAM MODULAR MAS — EXECUTION LOG ===")
print(result["log"])


=== TEAM MODULAR MAS — EXECUTION LOG ===

[research_team] → I was unable to retrieve the specific information from the web searches. However, I can provide a general overview based on the latest trends and information available up to 2023:

1. **Growth Rates**:
   - The electric vehicle market has been experiencing significant growth, driven by increasing environmental awareness, government incentives, and advancements in battery technology. The global EV market is expected to continue its rapid expansion, with some reports predicting a compound annual growth rate (CAGR) of over 20% in the coming years.

2. **Key Players**:
   - Major automotive companies like Tesla, BYD, Volkswagen, and General Motors are leading the charge in the EV market. These companies are investing heavily in EV technology and expanding their electric vehicle lineups. New entrants and startups are also emerging, contributing to the competitive landscape.

3. **Emerging Technologies**:
   - Innovations in batter

## 💡 Key Takeaways

- **Two-level structured routing** (coordinator → team supervisor → agent): each level makes a smaller, more focused decision.
- Both routing levels use **`llm.with_structured_output(Pydantic)`** — no string parsing, no fallbacks needed.
- Adding an agent to a team: create it and add it to `TEAMS[team_name]["agents"]` — zero changes to the coordinator or other teams.
- Adding a new team: add it to `TEAMS` and mention it in `COORDINATOR_SYSTEM` — zero changes to existing teams or agents.
- Team supervisors can be replaced with their own sub-graphs for arbitrarily deep nesting.